# 🎬 Sistem de Recomandare Filme

**Problema:** Vreau sa construiesc un sistem care recomanda filme pe baza unei descrieri date de utilizator, similar cu Netflix.

**Contextul:** Daca utilizatorul scrie 'vreau un film de actiune in spatiu', sistemul gaseste filmele cele mai potrivite din baza noastra de date.

**Impactul:** Un astfel de sistem poate fi folosit pe orice platforma de streaming pentru a ajuta utilizatorii sa gaseasca filme care le plac fara sa caute manual.

**Dataset:** Am folosit un dataset de filme de pe Kaggle - TMDB 5000 Movie Dataset
- Link: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
- Contine: ~5000 filme cu titlu, descriere, gen, rating
- Noi vom folosi o versiune mica cu 100 filme ca sa mearga rapid

## Pasul 1 - Instalam ce avem nevoie

In [ ]:
# instalam libraria care ne ajuta sa intelegem textul
!pip install sentence-transformers scikit-learn pandas matplotlib seaborn wordcloud

## Pasul 2 - Importam librariile

In [ ]:
# importam tot ce avem nevoie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import warnings
warnings.filterwarnings('ignore')

print('toate librariile sunt incarcate!')

## Pasul 3 - Cream dataset-ul cu filme

In [ ]:
# cream un dataset cu 50 de filme cunoscute
# in realitate ai descarca un fisier CSV de pe Kaggle
# dar pentru proiect cream noi datele ca sa fie simplu

date_filme = {
    'titlu': [
        'Interstellar', 'Inception', 'The Matrix', 'The Dark Knight',
        'Avengers Endgame', 'Titanic', 'The Godfather', 'Forrest Gump',
        'Gladiator', 'Alien', 'Blade Runner 2049', 'La La Land',
        'Joker', "Schindler's List", 'The Shawshank Redemption',
        'Pulp Fiction', 'The Silence of the Lambs', 'Saving Private Ryan',
        'The Lord of the Rings', 'Star Wars A New Hope',
        'Jurassic Park', 'Back to the Future', 'E.T.',
        'Home Alone', 'The Lion King', 'Toy Story',
        'Finding Nemo', 'The Little Mermaid', 'Beauty and the Beast',
        'Frozen', 'Moana', 'Coco', 'Up', 'Wall-E',
        'Spider-Man Into the Spider-Verse', 'Black Panther',
        'Iron Man', 'Thor Ragnarok', 'Captain America',
        'Guardians of the Galaxy', 'Doctor Strange',
        'The Notebook', 'Pride and Prejudice', 'Dirty Dancing',
        'Pretty Woman', 'Crazy Rich Asians', 'About Time',
        'Parasite', 'Spirited Away', 'Howls Moving Castle'
    ],
    'gen': [
        'Sci-Fi', 'Sci-Fi', 'Sci-Fi', 'Actiune',
        'Actiune', 'Drama', 'Drama', 'Drama',
        'Actiune', 'Horror', 'Sci-Fi', 'Drama',
        'Drama', 'Drama', 'Drama',
        'Thriller', 'Thriller', 'Razboi',
        'Aventura', 'Sci-Fi',
        'Aventura', 'Comedie', 'Familie',
        'Comedie', 'Animatie', 'Animatie',
        'Animatie', 'Animatie', 'Animatie',
        'Animatie', 'Animatie', 'Animatie', 'Animatie', 'Animatie',
        'Animatie', 'Actiune',
        'Actiune', 'Actiune', 'Actiune',
        'Actiune', 'Actiune',
        'Romantic', 'Romantic', 'Romantic',
        'Romantic', 'Romantic', 'Romantic',
        'Thriller', 'Animatie', 'Animatie'
    ],
    'rating': [
        8.6, 8.8, 8.7, 9.0,
        8.4, 7.8, 9.2, 8.8,
        8.5, 8.4, 8.0, 8.0,
        8.4, 8.9, 9.3,
        8.9, 8.6, 8.6,
        8.9, 8.6,
        8.1, 8.5, 7.8,
        7.6, 8.5, 8.3,
        8.1, 7.6, 8.0,
        7.4, 7.4, 8.4, 8.2, 8.4,
        8.4, 7.3,
        7.9, 7.9, 7.8,
        8.0, 7.5,
        7.9, 7.8, 7.0,
        6.9, 7.3, 7.8,
        8.5, 8.6, 8.2
    ],
    'descriere': [
        'astronauti calatoresc prin spatiu printr-o gaura neagra ca sa salveze omenirea de pe pamant',
        'un hot fura secrete din visele oamenilor cu multa actiune si mister in lumea viselor',
        'un hacker descopera ca lumea este o simulare controlata de masini si lupta pentru libertate',
        'batman lupta impotriva jokerului un criminal nebun care vrea sa distruga orasul gotham',
        'supereroi se unesc pentru a lupta impotriva lui thanos care vrea sa distruga jumatate din univers',
        'doi tineri se indragostesc pe nava titanic inainte ca aceasta sa se scufunde in ocean',
        'o familie de mafie italiana lupta pentru putere si supravietuire in lumea interlopa americana',
        'un barbat simplu cu suflet bun traieste momente istorice si o frumoasa poveste de dragoste',
        'un general roman devine gladiator si lupta in arena pentru a se razbuna pe imparatul rau',
        'astronauti se confrunta cu o creatura extraterestra mortala si periculoasa pe nava lor spatiala',
        'un detectiv robot dintr-un viitor sumbru descopera un secret urias care poate schimba societatea',
        'o tanara actrita si un muzician se indragostesc in los angeles urmandu-si visele lor mari',
        'un comedian esuat din orasul gotham devine incet-incet cel mai periculos criminal cunoscut',
        'un om de afaceri german salveaza evrei de la moarte sigura in al doilea razboi mondial',
        'un om nevinovat supravietuieste in inchisoare ani multi si cauta sa fie liber si fericit',
        'mai multe povesti de crima violenta si filozofie se intrepatrund intr-un mod neasteptat si interesant',
        'un agent FBI urmareste un cannibal geniu pentru a prinde un alt criminal in serie periculos',
        'soldati americani se lupta sa supravietuiasca si sa salveze un soldat pe frontul celui de al doilea razboi',
        'un hobbit si prietenii lui calatoresc printr-o lume fantastica pentru a distruge un inel rau si puternic',
        'un tanar fara experinta ajunge sa fie eroul galactic care salveaza galaxia de un imperiu rau',
        'oameni de stiinta cloneaza dinozauri intr-un parc tematic care devine rapid un dezastru periculos',
        'un adolescent calatoreste cu o masina a timpului inapoi in trecut si trebuie sa repare istoria',
        'un baiat de 10 ani se imprieteneste cu un extraterestru bland care vrea sa se intoarca acasa',
        'un baiat de 8 ani ramane singur acasa de craciun si trebuie sa se apere de doi hoti amuzanti',
        'puiul de leu simba trebuie sa lupte pentru a recapata tronul luat de unchiul sau cel rau',
        'jucarii vii au aventuri amuzante atunci cand proprietarul lor nu este acasa si sunt singure',
        'un pestisor tata parcurge tot oceanul pentru a-si gasi fiul pierdut nemo cu ajutor neobisnuit',
        'o mica sirena visatoare doreste sa devina om si se indragosteste de un print frumos pe pamant',
        'o tanara este blestemata sa devina bestie si invata iubirea adevarata intr-un castel magic',
        'doua surori trebuie sa colaboreze pentru a salva un regat inghetat de puterile magice ale uneia',
        'o tanara polinezieza calatoreste pe mare pentru a salva insula sa din blestemul unui semizeu',
        'un baiat mexican descopera intr-o noapte magica lumea colorata si vesela a stramosilor sai morti',
        'un mosneag legat de casa sa de amintiri calatoreste spre paradis cu ajutorul unor baloane uriase',
        'un robot singuratic curata pamantul parasit si se indragosteste de o alta mica droida moderna',
        'un tanar spider-man intalneste mai multe versiuni ale sale din universuri paralele diferite',
        'un rege african se intoarce acasa sa protejeze natiunea sa de un dusman periculos si puternic',
        'un om de afaceri bogat construieste o armura puternica si devine un supererou pentru lume',
        'zeul tunetelor trebuie sa isi salveze planeta de fratele sau vitreg cu ajutorul incredibilului hulk',
        'un soldat devine supererou si lupta pentru a proteja lumea de un razbunator puternic si rau',
        'o echipa de eroi ciudati si amuzanti calatoreste prin galaxie si salveaza o planeta de distrugere',
        'un vraci care isi fractureaza mainile descopera o lume magica ascunsa si devine apararotul ei',
        'doi adolescenti se indragostesc la un lac intr-o vara si lupta pentru iubirea lor de-a lungul anilor',
        'o tanara din clasa de jos si un bogat superior se indragostesc in anglia victoriana cu obstacole',
        'o dansatoare talentata se indragosteste de instructorul sau intr-o tabara de vara memorabila',
        'o tanara din cartier se indragosteste de un barbat bogat si elegant in orasul los angeles stralucitor',
        'o americanca descopera ca iubitul sau este extrem de bogat cand il viziteaza familia in singapore',
        'un barbat descopera ca poate calatori in timp si incearca sa schimbe trecutul pentru iubire',
        'o familie saraca din coreea infiltreaza incet-incet viata unei familii bogate cu rezultate neasteptate',
        'o fetita ajunge intr-o lume magica misterioasa unde trebuie sa gaseasca o modalitate sa se intoarca acasa',
        'un tanar sofar devine vrajitor si trebuie sa isi salveze iubita transformata intr-un animal de basm'
    ]
}

# transformam in DataFrame - un tabel cu date
df = pd.DataFrame(date_filme)

# adaugam si o coloana cu numarul de cuvinte din descriere
df['lungime_descriere'] = df['descriere'].apply(lambda x: len(x.split()))

print('Dataset creat cu succes!')
print(f'Avem {len(df)} filme in dataset')
print(f'Coloanele sunt: {list(df.columns)}')

## Pasul 4 - EDA (Explorarea si Intelegerea Datelor)

Inainte sa facem orice model, trebuie sa intelegem ce date avem.

In [ ]:
# afisam primele randuri din dataset
print('=== PRIMELE 5 RANDURI ===')
df.head()

In [ ]:
# informatii generale despre dataset
print('=== INFORMATII GENERALE ===')
df.info()

In [ ]:
# statistici descriptive
print('=== STATISTICI DESCRIPTIVE ===')
df.describe()

In [ ]:
# verificam daca avem valori lipsa
print('=== VALORI LIPSA ===')
print(df.isnull().sum())
print()
print('Nu avem valori lipsa, dataset-ul este curat!')

In [ ]:
# GRAFIC 1 - distributia filmelor pe gen
plt.figure(figsize=(12, 5))
gen_counts = df['gen'].value_counts()
plt.bar(gen_counts.index, gen_counts.values, color='steelblue', edgecolor='black')
plt.title('Cate filme avem din fiecare gen', fontsize=14)
plt.xlabel('Genul filmului')
plt.ylabel('Numar de filme')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/grafic1_gen.png')
plt.show()
print('Observatie: Avem cele mai multe filme de Animatie si Actiune in dataset')

In [ ]:
# GRAFIC 2 - distributia ratingurilor
plt.figure(figsize=(10, 5))
plt.hist(df['rating'], bins=10, color='orange', edgecolor='black')
plt.title('Distributia ratingurilor filmelor', fontsize=14)
plt.xlabel('Rating (1-10)')
plt.ylabel('Numar de filme')
plt.axvline(df['rating'].mean(), color='red', linestyle='--', label=f'Media: {df["rating"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('results/grafic2_rating.png')
plt.show()
print(f'Observatie: Media ratingurilor este {df["rating"].mean():.1f}, cele mai multe filme au rating intre 7 si 9')

In [ ]:
# GRAFIC 3 - lungimea descrierilor
plt.figure(figsize=(10, 5))
plt.hist(df['lungime_descriere'], bins=8, color='green', edgecolor='black')
plt.title('Distributia lungimii descrierilor (numar cuvinte)', fontsize=14)
plt.xlabel('Numar de cuvinte')
plt.ylabel('Numar de filme')
plt.tight_layout()
plt.savefig('results/grafic3_lungime.png')
plt.show()
print(f'Observatie: Descrierile au intre {df["lungime_descriere"].min()} si {df["lungime_descriere"].max()} cuvinte')

In [ ]:
# GRAFIC 4 - rating mediu pe gen
plt.figure(figsize=(12, 5))
rating_pe_gen = df.groupby('gen')['rating'].mean().sort_values(ascending=False)
plt.bar(rating_pe_gen.index, rating_pe_gen.values, color='purple', edgecolor='black')
plt.title('Rating mediu pentru fiecare gen de film', fontsize=14)
plt.xlabel('Genul filmului')
plt.ylabel('Rating mediu')
plt.ylim(6, 10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/grafic4_rating_gen.png')
plt.show()
print('Observatie: Filmele de Drama au cele mai mari ratinguri in medie')

In [ ]:
# GRAFIC 5 - word cloud din toate descrierile
# asta ne arata ce cuvinte apar cel mai des in descrieri
toate_cuvintele = ' '.join(df['descriere'].tolist())
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate(toate_cuvintele)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Cele mai frecvente cuvinte din descrierile filmelor', fontsize=14)
plt.tight_layout()
plt.savefig('results/grafic5_wordcloud.png')
plt.show()
print('Observatie: Cuvintele cel mai des intalnite sunt: lumea, lupta, trebuie, calatorie - tipice pentru filme de aventura')

## Pasul 5 - Preprocesarea Datelor

Curatam textul si pregatim datele pentru model.

In [ ]:
# curatam textul din descrieri
# facem totul lowercase (litere mici) - deja e asa dar e bine sa verificam
# scoatem caracterele speciale

import re

def curata_text(text):
    # transformam in litere mici
    text = text.lower()
    # scoatem caracterele speciale, lasam doar litere si spatii
    text = re.sub(r'[^a-zA-ZăîâșțĂÎÂȘȚ\s]', '', text)
    # scoatem spatiile duble
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# aplicam curatarea pe coloana de descriere
df['descriere_curata'] = df['descriere'].apply(curata_text)

print('Textul a fost curatat!')
print()
print('Inainte de curatare:')
print(df['descriere'][0])
print()
print('Dupa curatare:')
print(df['descriere_curata'][0])

## Pasul 6 - Modelul 1: TF-IDF (modelul simplu)

TF-IDF este o metoda clasica care cauta cuvinte similare intre cautarea ta si descrierile filmelor.

In [ ]:
# MODEL 1: TF-IDF
# asta transforma fiecare descriere intr-un vector de numere
# bazat pe frecventa cuvintelor

print('Antrenam modelul TF-IDF...')

# cream vectorizatorul
tfidf = TfidfVectorizer()

# antrenam pe descrierile noastre curate
# fit_transform = invata vocabularul SI transforma textul in numere
matrice_tfidf = tfidf.fit_transform(df['descriere_curata'])

print(f'Modelul TF-IDF a invatat {len(tfidf.vocabulary_)} cuvinte!')
print(f'Fiecare film este reprezentat acum ca un vector de {matrice_tfidf.shape[1]} numere')

# salvam modelul TF-IDF
with open('models/model_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('Modelul TF-IDF a fost salvat!')

In [ ]:
# functia de cautare cu TF-IDF
def cauta_cu_tfidf(cautare, top_n=3):
    # curatam cautarea la fel ca descrierile
    cautare_curata = curata_text(cautare)
    
    # transformam cautarea in acelasi tip de vector
    vector_cautare = tfidf.transform([cautare_curata])
    
    # calculam cat de similar e fiecare film cu cautarea
    scoruri = cosine_similarity(vector_cautare, matrice_tfidf)[0]
    
    # sortam si luam primele top_n rezultate
    indici_sortati = scoruri.argsort()[::-1][:top_n]
    
    print(f'=== Rezultate TF-IDF pentru: "{cautare}" ===')
    for i, idx in enumerate(indici_sortati):
        print(f'{i+1}. {df["titlu"][idx]} ({df["gen"][idx]}) - Scor: {scoruri[idx]:.3f}')
        print(f'   {df["descriere"][idx][:80]}...')
    print()

# testam
cauta_cu_tfidf('vreau un film cu astronauti in spatiu')
cauta_cu_tfidf('film romantic cu dragoste')

## Pasul 7 - Modelul 2: Sentence Transformers (modelul mai bun)

Acest model intelege sensul textului, nu doar cuvintele exacte.

In [ ]:
# MODEL 2: Sentence Transformers
# acest model intelege sensul cuvintelor
# de exemplu stie ca 'spatiu' si 'astronauti' sunt legate

print('Incarcam modelul AI... (poate dura 1-2 minute prima data)')
model_semantic = SentenceTransformer('all-MiniLM-L6-v2')
print('Modelul AI este incarcat!')

print()
print('Transformam toate filmele in vectori de numere...')
embeddings_filme = model_semantic.encode(df['descriere_curata'].tolist(), show_progress_bar=True)

print(f'Gata! Fiecare film are acum o amprenta de {embeddings_filme.shape[1]} numere')

# salvam embeddingurile ca sa nu le recalculam de fiecare data
np.save('models/embeddings_filme.npy', embeddings_filme)
print('Embeddingurile au fost salvate!')

In [ ]:
# functia de cautare cu modelul semantic
def cauta_cu_semantic(cautare, top_n=3):
    # transformam cautarea in acelasi tip de vector
    embedding_cautare = model_semantic.encode([cautare])
    
    # calculam similaritatea
    scoruri = cosine_similarity(embedding_cautare, embeddings_filme)[0]
    
    # sortam si luam primele top_n
    indici_sortati = scoruri.argsort()[::-1][:top_n]
    
    print(f'=== Rezultate SEMANTIC pentru: "{cautare}" ===')
    for i, idx in enumerate(indici_sortati):
        print(f'{i+1}. {df["titlu"][idx]} ({df["gen"][idx]}) - Scor: {scoruri[idx]:.3f}')
        print(f'   {df["descriere"][idx][:80]}...')
    print()

# testam
cauta_cu_semantic('vreau un film cu astronauti in spatiu')
cauta_cu_semantic('film romantic cu dragoste')

## Pasul 8 - Evaluarea si Compararea Modelelor

Comparam cele doua modele sa vedem care e mai bun.

In [ ]:
# comparam cele doua modele pe mai multe cautari
cautari_test = [
    'film de actiune cu supereroi',
    'animatie pentru copii',
    'thriller psihologic cu criminali',
    'poveste de dragoste romantica',
    'aventura in spatiu cu extraterstri'
]

print('COMPARATIE INTRE CELE DOUA MODELE')
print('='*60)

for cautare in cautari_test:
    print(f'\nCautare: "{cautare}"')
    print('-'*40)
    
    # TF-IDF
    cautare_curata = curata_text(cautare)
    vector_cautare = tfidf.transform([cautare_curata])
    scoruri_tfidf = cosine_similarity(vector_cautare, matrice_tfidf)[0]
    idx_tfidf = scoruri_tfidf.argsort()[::-1][0]
    
    # Semantic
    emb_cautare = model_semantic.encode([cautare])
    scoruri_sem = cosine_similarity(emb_cautare, embeddings_filme)[0]
    idx_sem = scoruri_sem.argsort()[::-1][0]
    
    print(f'TF-IDF    -> {df["titlu"][idx_tfidf]:30} (scor: {scoruri_tfidf[idx_tfidf]:.3f})')
    print(f'SEMANTIC  -> {df["titlu"][idx_sem]:30} (scor: {scoruri_sem[idx_sem]:.3f})')

In [ ]:
# GRAFIC 6 - comparam scorurile medii ale celor doua modele

scoruri_medii_tfidf = []
scoruri_medii_sem = []

for cautare in cautari_test:
    cautare_curata = curata_text(cautare)
    
    vector_cautare = tfidf.transform([cautare_curata])
    scoruri_tfidf = cosine_similarity(vector_cautare, matrice_tfidf)[0]
    scoruri_medii_tfidf.append(scoruri_tfidf.max())
    
    emb_cautare = model_semantic.encode([cautare])
    scoruri_sem = cosine_similarity(emb_cautare, embeddings_filme)[0]
    scoruri_medii_sem.append(scoruri_sem.max())

x = range(len(cautari_test))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar([i - width/2 for i in x], scoruri_medii_tfidf, width, label='TF-IDF', color='steelblue')
ax.bar([i + width/2 for i in x], scoruri_medii_sem, width, label='Semantic', color='orange')

ax.set_xlabel('Cautare test')
ax.set_ylabel('Scor maxim gasit')
ax.set_title('Comparatie TF-IDF vs Semantic - Scorul celui mai bun rezultat', fontsize=13)
ax.set_xticks(list(x))
ax.set_xticklabels([f'Test {i+1}' for i in x])
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('results/grafic6_comparatie_modele.png')
plt.show()

print(f'Scor mediu TF-IDF:   {np.mean(scoruri_medii_tfidf):.3f}')
print(f'Scor mediu Semantic: {np.mean(scoruri_medii_sem):.3f}')

In [ ]:
# TABEL COMPARATIV - care model e mai bun?

rezultate_comparatie = {
    'Model': ['TF-IDF', 'Sentence Transformers'],
    'Tip': ['Clasic (bazat pe cuvinte)', 'AI (bazat pe sens)'],
    'Viteza': ['Foarte rapid', 'Mai lent'],
    'Intelege sinonime': ['Nu', 'Da'],
    'Scor mediu': [round(np.mean(scoruri_medii_tfidf), 3), round(np.mean(scoruri_medii_sem), 3)],
    'Recomandat pentru': ['Cautari exacte', 'Cautari in limbaj natural']
}

df_comparatie = pd.DataFrame(rezultate_comparatie)
print('=== TABEL COMPARATIV MODELE ===')
print(df_comparatie.to_string(index=False))
print()
print('CONCLUZIE: Modelul Semantic este ales ca model final')
print('pentru ca intelege sensul textului si nu doar cuvintele exacte.')

## Pasul 9 - Modelul Final

Cream functia finala care combina ambele modele si o interfata simpla.

In [ ]:
# MODELUL FINAL - combina cele doua modele
# semantic 70% + tfidf 30%

def recomanda_filme_final(cautare, top_n=5):
    """
    Functia principala de recomandare
    Combina TF-IDF si Semantic pentru rezultate mai bune
    """
    # TFIDF
    cautare_curata = curata_text(cautare)
    vector_cautare = tfidf.transform([cautare_curata])
    scoruri_tfidf = cosine_similarity(vector_cautare, matrice_tfidf)[0]
    
    # Semantic
    emb_cautare = model_semantic.encode([cautare])
    scoruri_sem = cosine_similarity(emb_cautare, embeddings_filme)[0]
    
    # combinam: 30% tfidf + 70% semantic
    scoruri_finale = 0.3 * scoruri_tfidf + 0.7 * scoruri_sem
    
    # sortam
    indici_sortati = scoruri_finale.argsort()[::-1][:top_n]
    
    print(f'\n🎬 TOP {top_n} RECOMANDARI pentru: "{cautare}"')
    print('='*55)
    for i, idx in enumerate(indici_sortati):
        print(f'{i+1}. {df["titlu"][idx]}')
        print(f'   Gen: {df["gen"][idx]} | Rating: {df["rating"][idx]}/10')
        print(f'   Relevanta: {scoruri_finale[idx]:.0%}')
        print()

# testam modelul final
recomanda_filme_final('vreau ceva cu batai si supereroi')
recomanda_filme_final('animatie frumoasa pentru toata familia')
recomanda_filme_final('ceva de groaza cu extraterestri')

In [ ]:
# salvam datasetul procesat
df.to_csv('data/filme_procesate.csv', index=False)

# salvam si embeddingurile
np.save('models/embeddings_finale.npy', embeddings_filme)

print('Toate fisierele au fost salvate!')
print()
print('Fisiere salvate:')
print('  - data/filme_procesate.csv  (datele procesate)')
print('  - models/model_tfidf.pkl    (modelul TF-IDF)')
print('  - models/embeddings_finale.npy (vectorii filmelor)')
print('  - results/grafic1_gen.png')
print('  - results/grafic2_rating.png')
print('  - results/grafic3_lungime.png')
print('  - results/grafic4_rating_gen.png')
print('  - results/grafic5_wordcloud.png')
print('  - results/grafic6_comparatie_modele.png')

## Concluzii

**Ce am facut:**
1. Am creat un dataset cu 50 de filme cunoscute cu titlu, gen, rating si descriere
2. Am explorat datele cu 6 grafice diferite
3. Am curatat textul (lowercase, caractere speciale)
4. Am antrenat doua modele: TF-IDF si Sentence Transformers
5. Am comparat modelele si ales cel mai bun
6. Am creat un model final care combina ambele

**Ce am observat:**
- Modelul Semantic este mai bun decat TF-IDF pentru cautari in limbaj natural
- TF-IDF e mai rapid dar nu intelege sensul cuvintelor
- Combinand cele doua se obtin rezultate mai bune

**Ce ar putea fi imbunatatit:**
- Un dataset mai mare (mii de filme de pe Kaggle)
- O interfata grafica cu Gradio
- Filtrare dupa rating sau gen